## 1. Library 불러오기

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

from xgboost import XGBClassifier

## 2. 데이터 불러오기

In [ ]:
train=pd.read_csv('security_log_train.csv')
test=pd.read_csv('security_log_test.csv')
valid=pd.read_csv('sample_submission.csv')

In [3]:
train.head()

,id,level,full_log
0,0,0,"Sep 24 10:02:22 localhost kibana: {""type"":""err..."
1,1,0,Feb 8 16:21:00 localhost logstash: [2021-02-0...
2,2,0,"Jan 13 01:50:40 localhost kibana: {""type"":""err..."
3,3,0,"Jan 4 10:18:31 localhost kibana: {""type"":""err..."
4,4,1,type=SYSCALL msg=audit(1603094402.016:52981): ...


In [4]:
test.head()

,id,full_log
0,1000000,"Feb 8 15:47:26 localhost kibana: {""type"":""err..."
1,1000001,"Sep 24 03:46:39 localhost kibana: {""type"":""err..."
2,1000002,type=SYSCALL msg=audit(1611888200.428:210563):...
3,1000003,"Jan 18 11:24:06 localhost kibana: {""type"":""err..."
4,1000004,type=SYSCALL msg=audit(1603081202.050:46851): ...


In [5]:
valid.head()

,full_log
0,type=ANOM_PROMISCUOUS msg=audit(1600402733.466...
1,"oscap: msg: ""xccdf-result"", scan-id: ""00016007..."
2,Sep 22 10:56:19 localhost kernel: Out of memor...


In [6]:
train['level'].value_counts()

0    334065
1    132517
3      4141
5      2219
2        12
4        10
6         8
Name: level, dtype: int64

## 3. 전처리

In [7]:
# 숫자 마스킹 처리
train['full_log']=train['full_log'].str.replace(r'[0-9]', '<num>')
test['full_log']=test['full_log'].str.replace(r'[0-9]', '<num>')

In [ ]:
# Tfidf 기반 피쳐 벡터화 변환
tfidf_vect = TfidfVectorizer(min_df=0,ngram_range=(2,2)) 

text_mat = tfidf_vect.fit_transform(train['full_log'])

#print(text_mat.shape)
#print(tfidf_vect.vocabulary_)

## 4. Model - XGBoost

In [9]:
SEED=7

train_level=np.array(train['level'])

x_train, x_valid, y_train, y_valid = train_test_split(text_mat, train_level, test_size=0.2, stratify=train_level, random_state = SEED)

In [10]:
# XGBClassifier

xgb_clf = XGBClassifier(booster='gbtree', 
                    colsample_bylevel=0.8, 
                    colsample_bytree=0.7, 
                    gamma=0, 
                    max_depth=5, learning_rate=0.15,
                    n_estimators=100, 
                    nthread=4,
                    objective = 'multi:softmax',
                    silent= False,
                    random_state = SEED)

xgb_clf.fit(x_train, y_train, eval_set=[(x_valid, y_valid)],
             early_stopping_rounds=10)
xgb_clf.score(x_valid, y_valid)


[0]	validation_0-merror:0.004704
Will train until validation_0-merror hasn't improved in 10 rounds.
[1]	validation_0-merror:0.002188
[2]	validation_0-merror:0.00203
[3]	validation_0-merror:0.002061
[4]	validation_0-merror:0.002051
[5]	validation_0-merror:0.002019
[6]	validation_0-merror:0.001945
[7]	validation_0-merror:0.001998
[8]	validation_0-merror:0.001945
[9]	validation_0-merror:0.001945
[10]	validation_0-merror:0.001998
[11]	validation_0-merror:0.001945
[12]	validation_0-merror:0.001935
[13]	validation_0-merror:0.001935
[14]	validation_0-merror:0.001913
[15]	validation_0-merror:0.001913
[16]	validation_0-merror:0.001913
[17]	validation_0-merror:0.001913
[18]	validation_0-merror:0.001903
[19]	validation_0-merror:0.001903
[20]	validation_0-merror:0.001839
[21]	validation_0-merror:0.00185
[22]	validation_0-merror:0.00185
[23]	validation_0-merror:0.001808
[24]	validation_0-merror:0.001776
[25]	validation_0-merror:0.001755
[26]	validation_0-merror:0.001755
[27]	validation_0-merror:0.0

0.9982980072942544

In [11]:
#crosstab으로 확인
pred = xgb_clf.predict(x_valid)
crosstab = pd.crosstab(y_valid, pred, rownames=['real'], colnames=['pred'])
crosstab

pred,0,1,2,3,4,5,6
real,,,,,,,
0,66813,0,0,0,0,0,0
1,145,26359,0,0,0,0,0
2,0,0,2,0,0,0,0
3,0,0,0,828,0,0,0
4,0,0,0,0,2,0,0
5,16,0,0,0,0,428,0
6,0,0,0,0,0,0,2


In [12]:
from sklearn import metrics
metrics.f1_score(y_valid, pred, average='macro')

0.996815004491113

In [13]:
probas=xgb_clf.predict_proba(x_valid)
print(pred.shape)
print(probas.shape)

(94595,)
(94595, 7)


In [14]:
pd.DataFrame(probas)

,0,1,2,3,4,5,6
0,0.996777,0.000537,0.000531,0.000552,0.000531,0.000541,0.000531
1,0.996776,0.000537,0.000531,0.000552,0.000531,0.000542,0.000531
2,0.996777,0.000537,0.000531,0.000552,0.000531,0.000541,0.000531
3,0.996664,0.000537,0.000531,0.000628,0.000531,0.000578,0.000531
4,0.996776,0.000537,0.000531,0.000552,0.000531,0.000542,0.000531
...,...,...,...,...,...,...,...
94590,0.996777,0.000537,0.000531,0.000552,0.000531,0.000541,0.000531
94591,0.000517,0.996861,0.000518,0.000538,0.000518,0.000528,0.000518
94592,0.996776,0.000537,0.000531,0.000552,0.000531,0.000542,0.000531
94593,0.996777,0.000537,0.000531,0.000552,0.000531,0.000541,0.000531


## 5. level 7 Threshold 설정

In [15]:
# 각 행에서 가장 높은 확률을 뽑아 10분위 파악 
l=[]
l2=list(np.argmax(probas,axis=1))
for i in range(len(probas)):
    l.append(probas[i][l2[i]])
print(np.percentile(l,[0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1]))

[0.4246597  0.6351451  0.6522094  0.6556623  0.6556623  0.67380016
 0.95057148 0.96315759 0.97444284 0.97735404 0.98322105]


Threshold = 0.5 로 잡음

In [16]:
pred[np.where(np.max(probas, axis=1)<0.5)]=7
new_crosstab = pd.crosstab(y_valid, pred, rownames=['real'], colnames=['pred'])
new_crosstab

pred,0,1,2,3,4,5,6,7
real,,,,,,,,
0,66807,0,0,0,0,0,0,6
1,144,26359,0,0,0,0,0,1
2,0,0,2,0,0,0,0,0
3,0,0,0,828,0,0,0,0
4,0,0,0,0,2,0,0,0
5,15,0,0,0,0,428,0,1
6,0,0,0,0,0,0,2,0


## 6. Test

In [17]:
test_text=test['full_log']
test_text_mat=tfidf_vect.transform(test_text)

In [18]:
results=xgb_clf.predict(test_text_mat)
results_proba=xgb_clf.predict_proba(test_text_mat)
results[np.where(np.max(results_proba, axis=1) < 0.5)] = 7
results

array([0, 0, 1, ..., 1, 0, 0])

In [19]:
pd.Series(results).value_counts()

0    1004273
1     395119
3      12896
5       6292
7        247
4         34
2         34
6         21
dtype: int64

In [23]:
submission=pd.read_csv('/content/drive/MyDrive/(2021데이콘)로그분석/sample_submission.csv')
submission['level']=results
submission

,id,level
0,1000000,0
1,1000001,0
2,1000002,1
3,1000003,0
4,1000004,1
...,...,...
1418911,2418911,0
1418912,2418912,0
1418913,2418913,1
1418914,2418914,0
